# Nova AI — معالجة طابور الفيديو الحقيقي (CogVideoX-2B، ليس عرض شرائح)

**لماذا هذا الدفتر موجود:** owner spec 2026-09-12 ("لا اريد عرض شرائح...
اريد فديو حقيقي وليس شرائح"): توليد فيديو حقيقي بحركة فعلية (وليس
صوراً ثابتة بتأثير تكبير/تلاشي) يحتاج معالجاً رسومياً (GPU) — ModelScope
Studio (التشغيل الحي) بلا GPU إطلاقاً. الحل الصادق الوحيد بموارد مجانية:
هذا الدفتر يعمل بشكل مجدول (GitHub Actions، انظر
`.github/workflows/deploy-kaggle-video-queue.yml`) على GPU T4 المجاني
الحقيقي، يسحب طلبات فيديو من طابور `NovaVideoQueue` (جدول Supabase،
`prisma/migration_25_nova_video_queue.sql`)، يولّد فيديو حقيقياً فعلاً
(نفس CogVideoX-2B المُختبَر والمؤكَّد عمله سابقاً — 157 ثانية على T4
حقيقي، انظر خلية الاختبار في `generate_image_model.ipynb`)، ويُرسله
مباشرة إلى تيليجرام من هنا — لا حاجة لأي تخزين وسيط.

**فرق حقيقي مهم عن الوضع السابق:** هذا ليس فورياً — يعمل على دفعات
مجدولة (كل بضع ساعات)، فوصول الفيديو يأخذ ساعات لا دقائق. هذا تبادل
حقيقي (سرعة مقابل صدق التوليد) اختاره المالك صراحة بعد رؤية أن أسلوب
"الشرائح المتحركة" السابق لم يكن فيديو حقيقياً.

**ملاحظة معروفة من دفاتر هذا المشروع الأخرى:** دفع هذا الدفتر تلقائياً
عبر GitHub Actions قد يُعطّل أحياناً ربط أسرار Kaggle (Secrets) لهذا
الكيرنل بالتحديد، تماماً كما وُثِّق في الدفاتر الأخرى — تحقّق يدوياً من
تفعيل الأسرار على واجهة Kaggle بعد أول عملية دفع تلقائي.

In [ ]:
# الخلية 1 — تثبيت الأدوات
!pip install -q diffusers transformers accelerate torch requests imageio imageio-ffmpeg


In [ ]:
# الخلية 2 — أسرار Kaggle (نفس القيم المستخدمة في باقي دفاتر المشروع)
#
# أضف هذه كأسرار Kaggle (Add-ons -> Secrets) مرة واحدة:
# SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY (نفس مشروع Supabase الحقيقي)،
# NOVA_BOT_TOKEN (نفس توكن بوت نوفا على Render/Vercel)،
# NOVA_INTERNAL_SECRET (نفس القيمة المستخدمة أصلاً بين Ttbik وRender)،
# NOVA_FASTAPI_URL (رابط خادم Render، مثل https://nova-ai-backend-xxxx.onrender.com).
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
SUPABASE_URL = _secrets.get_secret("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = _secrets.get_secret("SUPABASE_SERVICE_ROLE_KEY")
NOVA_BOT_TOKEN = _secrets.get_secret("NOVA_BOT_TOKEN")
NOVA_INTERNAL_SECRET = _secrets.get_secret("NOVA_INTERNAL_SECRET")
NOVA_FASTAPI_URL = _secrets.get_secret("NOVA_FASTAPI_URL").rstrip("/")

_supabase_headers = {
    "apikey": SUPABASE_SERVICE_ROLE_KEY,
    "Authorization": f"Bearer {SUPABASE_SERVICE_ROLE_KEY}",
    "Content-Type": "application/json",
}

# أقصى عدد طلبات تُعالَج في تشغيل واحد — يحدّ الاستهلاك من حصة GPU
# المجانية الأسبوعية (30 ساعة، مشتركة مع دفتر التدريب الأسبوعي)،
# ويترك مجالاً حقيقياً لتشغيلات لاحقة بدل استنزاف الحصة كلها دفعة واحدة.
MAX_VIDEOS_PER_RUN = 5


In [ ]:
# الخلية 3 — تحميل نموذج الفيديو الحقيقي (نفس منطق الاختيار من generate_image_model.ipynb)
#
# نفس السببين الموثَّقين هناك: enable_model_cpu_offload/vae slicing
# وtiling ضرورية فعلياً لتشغيل CogVideoX-2B ضمن ذاكرة T4 المحدودة، لا
# تحسين اختياري.
import torch

video_pipe = None
VIDEO_KIND = None  # "cogvideox" | "text2video_ms"


def _try_cogvideox():
    from diffusers import CogVideoXPipeline

    pipe = CogVideoXPipeline.from_pretrained("THUDM/CogVideoX-2b", torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_slicing()
    pipe.vae.enable_tiling()
    return pipe, "cogvideox"


def _try_text2video_ms():
    from diffusers import DiffusionPipeline

    pipe = DiffusionPipeline.from_pretrained(
        "damo-vilab/text-to-video-ms-1.7b", torch_dtype=torch.float16, variant="fp16"
    )
    pipe.enable_model_cpu_offload()
    return pipe, "text2video_ms"


for _attempt in (_try_cogvideox, _try_text2video_ms):
    try:
        print("تجربة:", _attempt.__name__, "...")
        video_pipe, VIDEO_KIND = _attempt()
        print("نجح التحميل:", VIDEO_KIND)
        break
    except Exception as e:
        print("فشل", _attempt.__name__, "-", type(e).__name__, "-", str(e)[:200])

if video_pipe is None:
    raise RuntimeError("فشل تحميل كلا مرشَّحَي الفيديو — لا يمكن معالجة أي طلب في هذا التشغيل.")


In [ ]:
# الخلية 4 — سحب طلبات الطابور المعلَّقة (PENDING) ووسمها PROCESSING
#
# الوسم قبل المعالجة (لا بعدها) يمنع تشغيلاً متداخلاً لاحقاً (لو تأخر
# هذا التشغيل) من إعادة معالجة نفس الطلب مرتين.
import requests

resp = requests.get(
    f"{SUPABASE_URL}/rest/v1/NovaVideoQueue",
    headers=_supabase_headers,
    params={
        "select": "id,chatId,prompt,seconds",
        "status": "eq.PENDING",
        "order": "created_at.asc",
        "limit": str(MAX_VIDEOS_PER_RUN),
    },
    timeout=30,
)
resp.raise_for_status()
pending_rows = resp.json()
print(f"عدد الطلبات المعلَّقة المسحوبة لهذا التشغيل: {len(pending_rows)}")

for _row in pending_rows:
    requests.patch(
        f"{SUPABASE_URL}/rest/v1/NovaVideoQueue",
        headers=_supabase_headers,
        params={"id": f"eq.{_row['id']}"},
        json={"status": "PROCESSING"},
        timeout=30,
    )


In [ ]:
# الخلية 5 — المعالجة الفعلية: توليد حقيقي + إرسال مباشر لتيليجرام + تبليغ النتيجة
#
# كل طلب معزول في try/except خاص به — فشل طلب واحد (مثلاً prompt غير
# صالح) لا يجب أن يُسقط بقية الدفعة.
import traceback

from diffusers.utils import export_to_video

_VIDEO_FPS = 8  # يطابق افتراض _seconds_to_cogvideox_frames في council.py


def _seconds_to_frames(seconds: int) -> int:
    """نفس منطق council.py's _seconds_to_cogvideox_frames بالضبط —
    مكرَّر هنا عمداً (لا استيراد مشترك بين هذا الدفتر وخادم Render)،
    وليس تخميناً: قيد معماري حقيقي وموثَّق لـCogVideoX (4n+1 إطار)."""
    raw_frames = max(1, round(seconds * _VIDEO_FPS))
    n = round((raw_frames - 1) / 4)
    return max(25, n * 4 + 1)


def _report_result(queue_id: str, success: bool, error: str = None) -> None:
    try:
        requests.post(
            f"{NOVA_FASTAPI_URL}/admin/video-queue-result",
            headers={"X-Internal-Secret": NOVA_INTERNAL_SECRET},
            json={"queue_id": queue_id, "success": success, "error": (error or "")[:500]},
            timeout=30,
        )
    except Exception:
        print(f"تعذّر تبليغ نتيجة الطلب {queue_id} للخادم — سيبقى الطلب PROCESSING حتى مراجعة يدوية.")
        traceback.print_exc()


for _row in pending_rows:
    _qid, _chat_id, _prompt, _seconds = _row["id"], _row["chatId"], _row["prompt"], _row.get("seconds", 6)
    print(f"--- معالجة {_qid} (chat={_chat_id}): {_prompt[:80]}")
    try:
        _num_frames = _seconds_to_frames(_seconds)
        if VIDEO_KIND == "cogvideox":
            _frames = video_pipe(
                prompt=_prompt, num_videos_per_prompt=1, num_inference_steps=50,
                num_frames=_num_frames, guidance_scale=6,
            ).frames[0]
        else:
            _frames = video_pipe(prompt=_prompt, num_inference_steps=25, num_frames=16).frames[0]

        _out_path = f"/tmp/nova_video_{_qid}.mp4"
        export_to_video(_frames, _out_path, fps=_VIDEO_FPS)

        with open(_out_path, "rb") as _f:
            _send_resp = requests.post(
                f"https://api.telegram.org/bot{NOVA_BOT_TOKEN}/sendVideo",
                data={"chat_id": _chat_id, "caption": "🎬 فيديو حقيقي من Nova AI"},
                files={"video": (f"nova_{_qid}.mp4", _f, "video/mp4")},
                timeout=120,
            )
        if not _send_resp.ok:
            raise RuntimeError(f"Telegram sendVideo failed: {_send_resp.status_code} {_send_resp.text[:200]}")

        print("✅ تم التوليد والإرسال بنجاح.")
        _report_result(_qid, True)
    except Exception as e:
        print("❌ فشل هذا الطلب:", type(e).__name__, "-", str(e)[:300])
        traceback.print_exc()
        _report_result(_qid, False, error=f"{type(e).__name__}: {e}")

print("انتهت معالجة هذه الدفعة.")
